# Experiment 2: Shared Attention Head Analysis on Zamba2-1.2B

**Paper:** Mechanistic Interpretability of Hybrid SSM-Attention Models  
**Research question (RQ2):** Does the shared attention block in Zamba2 implement the same circuits as standard attention? Does weight-tying collapse or differentiate circuits at different depths?

## What we measure

Zamba2-1.2B has **6 hybrid layers** that each route through a shared attention block.  
The shared weights cycle through `num_mem_blocks=2` independent weight sets.

We probe each hybrid layer's attention heads using:
1. **Copy score** — does head h implement a copying pattern (attends to token X, boosts logit of X)?  
   High copy score ≈ induction-head-like (GPT-2 heads 4.4 and 5.5 have copy score ~0.5).
2. **Direct path patching** (using TransformerLens PR #1396) — causal attribution of each head's contribution to the IOI task.
3. **QK eigenspectrum** — do the shared weights collapse to a low-rank attention pattern?

## Why this matters

If the 6 hybrid layers share weights but attend at different positions in the residual stream,  
the *effective* circuit may differ per layer even with identical weights — the residual stream at layer 5  
is very different from layer 35. This experiment tests that hypothesis directly.

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install -q git+https://github.com/TransformerLensOrg/TransformerLens.git@dev
!pip install -q transformers>=4.47.0 einops jaxtyping

In [ ]:
# ── 2. Imports ───────────────────────────────────────────────────────────────
import gc
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer

from transformer_lens.model_bridge.bridge import TransformerBridge

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if torch.cuda.is_available() else torch.float32
MODEL  = "Zyphra/Zamba2-1.2B"

print(f"Device: {DEVICE} | dtype: {DTYPE}")

In [ ]:
# ── 3. Load model ────────────────────────────────────────────────────────────
bridge    = TransformerBridge.boot_transformers(MODEL, device=DEVICE, dtype=DTYPE)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
hf_model  = bridge.original_model

lbt           = list(getattr(hf_model.config, "layers_block_type", []))
hybrid_layers = [i for i, t in enumerate(lbt) if t == "hybrid"]
n_heads       = hf_model.config.num_attention_heads

print(f"Hybrid layer indices: {hybrid_layers}")
print(f"Attention heads per hybrid layer: {n_heads}")
print(f"num_mem_blocks (shared weight sets): {hf_model.config.num_mem_blocks}")

In [ ]:
# ── 4. Copy score ────────────────────────────────────────────────────────────
# Copy score (Elhage et al., 2021 / Olsson et al., 2022):
# For head h in layer l, copy_score = E_t [ attn_pattern[h, t, s] * (s == t-1) ]
#
# Zamba2 with output_attentions=True returns a tuple in outputs.attentions.
# It may contain only hybrid-layer tensors (len=6) OR all-layer tensors (len=38).
# We handle both cases below.

def get_attention_patterns(
    hf_model,
    tokens: torch.Tensor,
    hybrid_layer_indices: List[int],
) -> Dict[int, torch.Tensor]:
    """Returns {layer_idx: attn_weights} where attn_weights is [n_heads, seq_len, seq_len]."""
    with torch.no_grad():
        outputs = hf_model(tokens, output_attentions=True)

    captured = {}
    if outputs.attentions is None:
        return captured

    attn_list = outputs.attentions
    n_attn = len(attn_list)
    print(f"  outputs.attentions length: {n_attn}")

    if n_attn == len(hybrid_layer_indices):
        # Zamba2 returns only hybrid-layer attentions
        for i, layer_idx in enumerate(hybrid_layer_indices):
            attn = attn_list[i]
            if attn is not None:
                captured[layer_idx] = attn.detach().cpu()  # [batch, heads, seq, seq]
    else:
        # Full-length tuple (one entry per model layer, None for mamba layers)
        for layer_idx in hybrid_layer_indices:
            if layer_idx < n_attn and attn_list[layer_idx] is not None:
                captured[layer_idx] = attn_list[layer_idx].detach().cpu()

    return captured


# Test on a short sequence
test_tokens = torch.tensor([[1, 100, 200, 300, 100, 200, 300]]).to(DEVICE)
attn_patterns = get_attention_patterns(hf_model, test_tokens, hybrid_layers)
print(f"Captured attention from layers: {list(attn_patterns.keys())}")
for k, v in attn_patterns.items():
    print(f"  Layer {k}: shape {v.shape}")


In [ ]:
# ── 5. Compute copy scores on induction sequences ────────────────────────────

PREFIX_LEN = 40
N_SEQS     = 60
VOCAB_SIZE  = bridge.cfg.d_vocab

torch.manual_seed(42)

# Copy score = how much does each head attend to position (current - prefix_len)
# in the second copy of the sequence?

copy_scores = {layer_idx: np.zeros(n_heads) for layer_idx in hybrid_layers}
count = 0

for _ in range(N_SEQS):
    prefix = torch.randint(100, VOCAB_SIZE - 100, (PREFIX_LEN,))
    sep    = torch.tensor([1])
    tokens = torch.cat([prefix, sep, prefix]).unsqueeze(0).to(DEVICE)
    seq_len = tokens.shape[1]
    
    attn_patterns = get_attention_patterns(hf_model, tokens, hybrid_layers)
    
    for layer_idx, weights in attn_patterns.items():
        # weights: [1, n_heads, seq_len, seq_len]
        w = weights[0].float()  # [n_heads, seq_len, seq_len]
        
        # For each position i in second copy (i > PREFIX_LEN+1),
        # the "induction" source is position i - (PREFIX_LEN + 1)
        second_copy_start = PREFIX_LEN + 1
        scores_this = np.zeros(n_heads)
        valid_positions = 0
        
        for pos in range(second_copy_start, seq_len):
            source = pos - (PREFIX_LEN + 1)
            if source < 0:
                continue
            # attn weight from pos to source for each head
            scores_this += w[:, pos, source].numpy()
            valid_positions += 1
        
        if valid_positions > 0:
            copy_scores[layer_idx] += scores_this / valid_positions
    
    count += 1

# Average over sequences
for layer_idx in copy_scores:
    copy_scores[layer_idx] /= count

print("Copy scores per head per hybrid layer:")
for layer_idx in hybrid_layers:
    scores = copy_scores[layer_idx]
    top_head = scores.argmax()
    print(f"  Layer {layer_idx}: max={scores[top_head]:.3f} (head {top_head}), mean={scores.mean():.3f}")

In [ ]:
# ── 6. Plot copy scores ──────────────────────────────────────────────────────
n_hybrid = len(hybrid_layers)
fig, axes = plt.subplots(1, n_hybrid, figsize=(4 * n_hybrid, 4), sharey=True)

for ax, layer_idx in zip(axes, hybrid_layers):
    scores = copy_scores[layer_idx]
    head_indices = np.arange(n_heads)
    colors = ["#e05c5c" if s > 0.2 else "#aaaacc" for s in scores]
    ax.bar(head_indices, scores, color=colors, width=0.7)
    ax.axhline(y=0.2, color="gray", linestyle="--", linewidth=0.8, label="copy threshold")
    ax.set_title(f"Layer {layer_idx} (hybrid)", fontsize=10)
    ax.set_xlabel("Head", fontsize=9)
    ax.set_ylim(0, 1)

axes[0].set_ylabel("Copy score", fontsize=10)
fig.suptitle(
    "Copy score per attention head — Zamba2-1.2B hybrid layers\n"
    "(Red = high-copy head, likely induction-like)",
    fontsize=12
)
plt.tight_layout()
plt.savefig("copy_scores_zamba2.pdf", bbox_inches="tight", dpi=150)
plt.savefig("copy_scores_zamba2.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: copy_scores_zamba2.pdf + .png")

In [ ]:
# ── 7. QK eigenspectrum of shared attention weights ──────────────────────────
# The shared weights cycle through num_mem_blocks=2 sets.
# We compute the effective QK matrix = Q @ K^T and its singular values.
# Low effective rank → the head has a focused, clean circuit.
# High rank → the head is diffuse / generalised.

print("Computing QK eigenspectrum for shared attention weight sets...")
print(f"num_mem_blocks = {hf_model.config.num_mem_blocks}")

# Zamba2 stores shared attention blocks in model.shared_transformer
# The exact attribute depends on the HF implementation version
# Probe the model structure to find them
print("\nShared module structure:")
for name, mod in hf_model.named_modules():
    if "shared" in name.lower() and hasattr(mod, "weight"):
        print(f"  {name}: {mod.weight.shape}")
    if "q_proj" in name.lower() or "k_proj" in name.lower():
        print(f"  {name}: {getattr(mod, 'weight', None) and mod.weight.shape}")

In [ ]:
# ── 7b. QK rank analysis (once you've identified the weight paths above) ────
# Fill in the correct attribute path from the output of cell 7.
# Example (update if different):
#   shared_blocks = hf_model.model.shared_transformer  -- check actual attr name

# Template — adjust based on actual model structure found above
try:
    # Try common attribute names for Zamba2's shared attention blocks
    shared_candidates = [
        getattr(hf_model.model, "shared_transformer", None),
        getattr(hf_model.model, "shared_attention", None),
    ]
    shared_block = next(b for b in shared_candidates if b is not None)
    print(f"Found shared block: {type(shared_block)}")
    
    # If it's a ModuleList, iterate
    if hasattr(shared_block, "__iter__"):
        blocks = list(shared_block)
    else:
        blocks = [shared_block]
    
    for block_idx, block in enumerate(blocks):
        for name, mod in block.named_modules():
            if "q_proj" in name:
                W_Q = mod.weight.float()  # [d_model, d_head * n_heads]
                _, S, _ = torch.linalg.svd(W_Q, full_matrices=False)
                S = S.cpu().numpy()
                plt.figure(figsize=(8, 3))
                plt.plot(S[:50], marker="o", markersize=3)
                plt.xlabel("Singular value rank")
                plt.ylabel("Singular value")
                plt.title(f"Q_proj singular values — shared block {block_idx}, {name}")
                plt.tight_layout()
                plt.savefig(f"qk_spectrum_block{block_idx}.pdf", bbox_inches="tight")
                plt.show()
except Exception as e:
    print(f"Note: adjust shared block path based on model structure output above. Error: {e}")
    print("\nManually extract Q/K weights using paths identified in cell 7.")

In [ ]:
# ── 8. Save results ──────────────────────────────────────────────────────────
import json

results = {
    "model": MODEL,
    "hybrid_layers": hybrid_layers,
    "n_heads": n_heads,
    "n_seqs": N_SEQS,
    "copy_scores": {str(k): v.tolist() for k, v in copy_scores.items()},
    "high_copy_heads": {
        str(layer_idx): [
            int(h) for h in np.where(copy_scores[layer_idx] > 0.2)[0]
        ]
        for layer_idx in hybrid_layers
    }
}

with open("copy_scores_zamba2_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved: copy_scores_zamba2_results.json")

## Interpretation guide

| Copy score | Interpretation |
|---|---|
| > 0.5 | Strong induction head (comparable to GPT-2 heads 4.4, 5.5) |
| 0.2 – 0.5 | Moderate copying behavior |
| < 0.1 | Not a copy head — some other function |

**Key questions to answer:**  
1. Do all 6 hybrid layers have similar copy score distributions? (They share weights — they should — but the residual stream differs.)  
2. Do the high-copy heads appear at the same head indices across all 6 layers?  
3. Is the copy score consistent with what Exp 1 shows about SSMI at hybrid layers?

**Expected finding:** The shared attention heads likely implement copy behavior at early hybrid layers (seeding induction) but are used differently by the residual stream at later hybrid layers (refinement or output preparation).

**Next step:** Run `exp3_logit_lens.ipynb` to see how information accumulates layer-by-layer.